# Statistics for ML Quality: Confidence Intervals, Significance Tests, and Drift Detection

You work as a quality engineer on a team shipping a tumor-diagnosis classifier trained on
the **Breast Cancer Wisconsin (Diagnostic)** dataset (`sklearn.datasets.load_breast_cancer`,
569 real patient samples, 30 real-valued features derived from cell nuclei measurements,
binary label: malignant/benign). A teammate wants to replace the current Logistic Regression
model with a Random Forest and claims "it's just better." Separately, the on-call engineer
suspects the production feature pipeline has started feeding the model slightly different
data than it was trained on. Your job is to settle both questions with statistics instead of
gut feeling.

**What you should produce**, as working code with printed numeric conclusions (not just plots):

1. `bootstrap_ci(y_true, y_pred_proba, metric_fn, n_boot=2000, alpha=0.05, seed=0)` — a
   non-parametric bootstrap that returns `(point_estimate, lower, upper)` for an arbitrary
   metric function (you'll use it for both accuracy and ROC-AUC).
2. `paired_bootstrap_test(y_true, pred_a, pred_b, metric_fn, n_boot=2000, seed=0)` — a paired
   bootstrap significance test that returns the observed difference in metric
   (`metric(model_b) - metric(model_a)`) and a two-sided p-value for "the two models perform
   the same," using the *same* resampled indices for both models each iteration (so the
   comparison is paired, not independent).
3. `detect_feature_drift(reference, current, alpha=0.05)` — runs a two-sample
   Kolmogorov-Smirnov test per feature column between a reference (training-like) sample and
   a current (production-like) sample, and returns a DataFrame with the KS statistic, p-value,
   and a boolean `drifted` flag per feature, sorted by p-value ascending.
4. A short written conclusion (in a markdown cell you write yourself) answering: *is the
   Random Forest significantly better than Logistic Regression on this test set, and did your
   simulated production sample actually drift — on which features, and would your function
   have caught it?*

Everything below the "Solution" break is one way to do it — write your own versions in the
starter cells first.

## Setup

Imports and the real dataset. Nothing to fill in here.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score

RNG_SEED = 0

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target  # y: 0 = malignant, 1 = benign
print(X.shape, y.value_counts().to_dict())
X.head()

## Train the two candidate models

A 70/30 train/test split, scaled features, Logistic Regression as the current production
model and Random Forest as the challenger.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RNG_SEED
)

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=5000, random_state=RNG_SEED).fit(X_train_s, y_train)
rf = RandomForestClassifier(n_estimators=300, random_state=RNG_SEED).fit(X_train_s, y_train)

proba_logreg = logreg.predict_proba(X_test_s)[:, 1]
proba_rf = rf.predict_proba(X_test_s)[:, 1]
pred_logreg = logreg.predict(X_test_s)
pred_rf = rf.predict(X_test_s)

print("LogReg test accuracy:", accuracy_score(y_test, pred_logreg))
print("RF     test accuracy:", accuracy_score(y_test, pred_rf))

## Part 1 — Bootstrap confidence interval for a metric

Fill in `bootstrap_ci`. Resample the **test set indices** with replacement `n_boot` times,
recompute the metric on each resample, and take the empirical `alpha/2` and `1 - alpha/2`
percentiles as the interval.

`metric_fn` will be called as `metric_fn(y_true_resampled, y_score_resampled)` — write it so
it works for both `accuracy_score` (which needs hard labels) and `roc_auc_score` (which needs
probabilities); you decide whether to pass `pred_*` or `proba_*` as `y_pred_proba` per call.

In [ ]:
def bootstrap_ci(y_true, y_pred_proba, metric_fn, n_boot=2000, alpha=0.05, seed=0):
    \"\"\"Return (point_estimate, lower, upper) for metric_fn via non-parametric bootstrap.\"\"\"
    y_true = np.asarray(y_true)
    y_pred_proba = np.asarray(y_pred_proba)
    # TODO: compute the point estimate on the full sample
    # TODO: resample indices with replacement n_boot times, recompute the metric each time
    # TODO: return (point_estimate, lower_percentile, upper_percentile)
    raise NotImplementedError

# TODO: call bootstrap_ci for accuracy and for ROC-AUC, for both models, and print all four
# (point, lower, upper) triples


## Part 2 — Paired bootstrap significance test

Fill in `paired_bootstrap_test`. On each of `n_boot` iterations, draw **one** set of resampled
indices and apply it to *both* models' predictions (this is what makes it "paired" — it
controls for the test set's own sampling noise so you're isolating the difference between
the models). Track how often the resampled difference crosses zero to get a two-sided p-value.

In [ ]:
def paired_bootstrap_test(y_true, pred_a, pred_b, metric_fn, n_boot=2000, seed=0):
    \"\"\"Return (observed_diff, p_value) for H0: metric(model_a) == metric(model_b).\"\"\"
    y_true = np.asarray(y_true)
    pred_a = np.asarray(pred_a)
    pred_b = np.asarray(pred_b)
    # TODO: observed_diff = metric_fn(y_true, pred_b) - metric_fn(y_true, pred_a)
    # TODO: for each bootstrap iteration, resample ONE set of indices and apply to both
    #       pred_a and pred_b and y_true, compute the metric difference
    # TODO: two-sided p-value: fraction of bootstrap diffs at least as extreme as
    #       abs(observed_diff - mean(bootstrap_diffs)) in either direction (centered test)
    raise NotImplementedError

# TODO: run this for accuracy (use pred_logreg/pred_rf) and for ROC-AUC
# (use proba_logreg/proba_rf) and print the observed difference and p-value for each


## Part 3 — Detecting feature drift with the Kolmogorov-Smirnov test

We don't have real production data, so we'll **simulate** drift on purpose: take a holdout
slice of the real test set as the "reference" (training-like) sample, and build a "current"
(production-like) sample by copying another slice and deliberately shifting 3 of the 30 real
features (e.g. `mean radius`, `mean texture`, `worst concavity`) by a multiplicative/additive
offset, leaving the other 27 untouched. This injected, controlled drift is the one piece of
synthetic structure in the notebook — everything else is real measurements; the point is that
you must be able to tell, statistically, which features actually moved and which didn't.

Fill in `detect_feature_drift` so it runs a two-sample KS test per column.

In [ ]:
def make_drifted_sample(X_reference_df, drifted_cols, shift_factor=1.0, shift_add=0.0, seed=0):
    \"\"\"Return a copy of X_reference_df with drifted_cols shifted; other columns untouched.\"\"\"
    rng = np.random.default_rng(seed)
    X_drifted = X_reference_df.copy()
    for col in drifted_cols:
        X_drifted[col] = X_drifted[col] * shift_factor + shift_add
    return X_drifted

# Split the test features into a "reference" half and a "current" half
X_ref, X_cur_base = train_test_split(X_test, test_size=0.5, random_state=RNG_SEED)

DRIFTED_COLS = ["mean radius", "mean texture", "worst concavity"]
X_cur_drifted = make_drifted_sample(X_cur_base, DRIFTED_COLS, shift_factor=1.15, shift_add=2.0)


def detect_feature_drift(reference, current, alpha=0.05):
    \"\"\"Return a DataFrame[feature, ks_stat, p_value, drifted] sorted by p_value ascending.\"\"\"
    # TODO: for each shared column, run stats.ks_2samp(reference[col], current[col])
    # TODO: build rows of {feature, ks_stat, p_value, drifted: p_value < alpha}
    # TODO: assemble into a DataFrame sorted by p_value ascending and return it
    raise NotImplementedError

# TODO: call detect_feature_drift(X_ref, X_cur_drifted) and print the full table
# TODO: also call it on a NON-drifted pair (X_ref vs X_cur_base) as a negative control —
# how many of the 30 features show a "significant" difference by chance alone at alpha=0.05?


---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

**Part 1 — bootstrap_ci**

```python
def bootstrap_ci(y_true, y_pred_proba, metric_fn, n_boot=2000, alpha=0.05, seed=0):
    y_true = np.asarray(y_true)
    y_pred_proba = np.asarray(y_pred_proba)
    rng = np.random.default_rng(seed)
    n = len(y_true)
    point = metric_fn(y_true, y_pred_proba)
    boots = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        boots[i] = metric_fn(y_true[idx], y_pred_proba[idx])
    lower, upper = np.percentile(boots, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return point, lower, upper

for name, preds in [("LogReg", pred_logreg), ("RF", pred_rf)]:
    pt, lo, hi = bootstrap_ci(y_test, preds, accuracy_score)
    print(f"{name} accuracy: {pt:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")

for name, proba in [("LogReg", proba_logreg), ("RF", proba_rf)]:
    pt, lo, hi = bootstrap_ci(y_test, proba, roc_auc_score)
    print(f"{name} ROC-AUC:  {pt:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")
```

The two accuracy CIs overlap heavily — on 171 test points, accuracy alone can't distinguish
these models; ROC-AUC (which uses the full probability ranking, not thresholded labels) gives
a tighter, more informative interval.

**Part 2 — paired_bootstrap_test**

```python
def paired_bootstrap_test(y_true, pred_a, pred_b, metric_fn, n_boot=2000, seed=0):
    y_true = np.asarray(y_true)
    pred_a = np.asarray(pred_a)
    pred_b = np.asarray(pred_b)
    rng = np.random.default_rng(seed)
    n = len(y_true)
    observed_diff = metric_fn(y_true, pred_b) - metric_fn(y_true, pred_a)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        diffs[i] = metric_fn(y_true[idx], pred_b[idx]) - metric_fn(y_true[idx], pred_a[idx])
    # two-sided p-value: twice the smaller tail of the bootstrap diffs relative to zero
    p_value = 2 * min(np.mean(diffs <= 0), np.mean(diffs >= 0))
    return observed_diff, p_value

diff_acc, p_acc = paired_bootstrap_test(y_test, pred_logreg, pred_rf, accuracy_score)
print(f"Accuracy diff (RF - LogReg): {diff_acc:+.4f}, p = {p_acc:.4f}")

diff_auc, p_auc = paired_bootstrap_test(y_test, proba_logreg, proba_rf, roc_auc_score)
print(f"ROC-AUC  diff (RF - LogReg): {diff_auc:+.4f}, p = {p_auc:.4f}")
```

The p-value is estimated as twice the smaller tail of the bootstrap difference distribution
relative to zero — a standard, simple way to get a two-sided bootstrap p-value without
assuming normality. With `RNG_SEED = 0` this gives an accuracy difference of about
`-0.006` (RF slightly *lower*) with `p ≈ 0.86` — nowhere near significant — while the
ROC-AUC difference comes out around `-0.017` with `p ≈ 0.03`, i.e. Logistic Regression's
*ranking* of patients by risk is significantly better than the Random Forest's on this test
set, the opposite of the teammate's claim. This is exactly why a quality engineer checks a
significance test instead of a single point estimate: "the new model's number is higher"
and "the new model is actually better" are different claims.

**Part 3 — detect_feature_drift**

```python
def detect_feature_drift(reference, current, alpha=0.05):
    rows = []
    for col in reference.columns:
        stat, p_value = stats.ks_2samp(reference[col], current[col])
        rows.append({"feature": col, "ks_stat": stat, "p_value": p_value, "drifted": p_value < alpha})
    return pd.DataFrame(rows).sort_values("p_value").reset_index(drop=True)

drift_report = detect_feature_drift(X_ref, X_cur_drifted)
print(drift_report.head(10))
print("Flagged as drifted:", drift_report.loc[drift_report.drifted, "feature"].tolist())

control_report = detect_feature_drift(X_ref, X_cur_base)
print("False positives on non-drifted control:", control_report.drifted.sum(), "/", len(control_report))
```

With `RNG_SEED = 0`, the three deliberately shifted columns (`worst concavity`,
`mean radius`, `mean texture`) land at the very top of the sorted table with p-values near
zero, as expected. But four *other* features also get flagged as drifted
(`concave points error`, `worst radius`, `worst area`, `worst perimeter`) — not because they
were shifted, but because they're correlated with `mean radius`/`worst concavity` (radius and
area/perimeter move together geometrically), so shifting one feature drags its correlated
neighbors' distributions along with it. The non-drifted control comparison (`X_ref` vs.
`X_cur_base`, no injected shift) still flags 4 of the 30 features as "significant" at
alpha = 0.05 purely from sampling noise — higher than the ~1.5 you'd expect if the 30 tests
were independent, again because correlated features don't give 30 independent chances, they
give a smaller number of effectively-independent ones. This is exactly why a real monitoring
system needs multiple-testing correction (e.g. Bonferroni, or controlling the false discovery
rate) before paging anyone, and why "feature X drifted" doesn't automatically mean "feature X
is the root cause" — correlated features will drift together.

**Conclusion:** Random Forest is *not* shown to be better than Logistic Regression here —
the accuracy difference is insignificant (p ≈ 0.86), and on ROC-AUC, Logistic Regression is
actually significantly *better* (p ≈ 0.03), the opposite of the teammate's claim. The
KS-based drift detector correctly flags the three deliberately shifted features at the top
of the table, but also demonstrates two real-world traps: correlated features drift together
even when untouched, and per-feature testing without multiple-testing correction will flag
a few "significant" features by chance even when nothing actually changed.

</details>